In [1]:
# 1. Install Kaggle
!pip install -q kaggle


# 2. Configure Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
#Download  dataset
!kaggle datasets download -d salader/dogsvscats

Dataset URL: https://www.kaggle.com/datasets/salader/dogsvscats
License(s): unknown
100% 1.06G/1.06G [00:52<00:00, 22.0MB/s]

unzip:  cannot find or open dataset-name.zip, dataset-name.zip.zip or dataset-name.zip.ZIP.


In [3]:
#unzipping the dataset
!unzip -q dogsvscats.zip -d dataset

In [10]:
#Load the images
import tensorflow as tf

data = tf.keras.utils.image_dataset_from_directory(
    "dataset/catsvsdogs/train",
    image_size=(128, 128),
    batch_size=32
)

class_names = data.class_names
print(class_names)

Found 20000 files belonging to 2 classes.
['cats', 'dogs']


In [11]:
#Split into training and validation
train_size = int(0.8 * len(data))

train = data.take(train_size)
test = data.skip(train_size)

In [12]:
#Create a simple CNN
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

In [13]:
#Train
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train,
    validation_data=test,
    epochs=5
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.6465 - loss: 0.6341 - val_accuracy: 0.7045 - val_loss: 0.5814
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 68ms/step - accuracy: 0.7511 - loss: 0.5046 - val_accuracy: 0.7655 - val_loss: 0.5097
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - accuracy: 0.8029 - loss: 0.4274 - val_accuracy: 0.7707 - val_loss: 0.4773
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 58ms/step - accuracy: 0.8454 - loss: 0.3543 - val_accuracy: 0.7810 - val_loss: 0.4912
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 57ms/step - accuracy: 0.8774 - loss: 0.2845 - val_accuracy: 0.7745 - val_loss: 0.5382


In [14]:
#Check accuracy
loss, accuracy = model.evaluate(test)
print("Accuracy:", accuracy)

125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.7750 - loss: 0.5391
Accuracy: 0.7749999761581421


In [16]:
#Predict an image
from google.colab import files

uploaded = files.upload()

img = tf.keras.utils.load_img(
    list(uploaded.keys())[0],
    target_size=(128, 128)
)

img = tf.keras.utils.img_to_array(img)
img = tf.expand_dims(img, 0)

prediction = model.predict(img)

print("Predicted:", class_names[tf.argmax(prediction[0])])

Saving photo-1529778873920-4da4926a72c2.avif to photo-1529778873920-4da4926a72c2.avif
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Predicted: cats


In [17]:
# Predict an image
from google.colab import files

uploaded = files.upload()

img = tf.keras.utils.load_img(
    list(uploaded.keys())[0],
    target_size=(128, 128)
)

img = tf.keras.utils.img_to_array(img)
img = tf.expand_dims(img, 0)

prediction = model.predict(img)[0]

# Show probability of each class
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {prediction[i] * 100:.2f}%")

# Final prediction
print("Predicted:", class_names[tf.argmax(prediction)])

Saving funny-portrait-of-cute-corgi-dog-outdoors-free-photo.jpg to funny-portrait-of-cute-corgi-dog-outdoors-free-photo (1).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
cats: 5.16%
dogs: 94.84%
Predicted: dogs
